Imports & Loading

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Ensure we can import from the src directory
import sys
sys.path.append('..')

# Import our modular logic
from src.processor import DataProcessor
from src.models import EnsembleWrapper
from src.conformal import get_residuals, apply_enbpi, rolling_agaci
from src.visualization import plot_forecast_vs_actual, plot_prediction_intervals, calculate_metrics

# Load Project Configuration
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set plotting style
plt.style.use('ggplot')
%matplotlib inline

Exploratory Data Analysis (EDA)

In [ ]:
# 1. Load Data
df = pd.read_csv(f"../{config['data']['file_path']}")
col = config['data']['target_column']

# Ensure numeric (fixes '^GSPC' / string issues)
df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop invalid values
df = df.dropna(subset=[col])

# Convert and set datetime index
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.set_index('Date', inplace=True)

# 2. Visualizing the Target
plt.figure(figsize=(14, 6))
plt.plot(df[config['data']['target_column']], color='#2c3e50')
plt.title(f"Historical {config['data']['target_column']} Price", fontsize=15)
plt.ylabel("Price (USD)")
plt.grid(True, alpha=0.3)
plt.show()

# 3. Volatility Analysis (Returns)
df['Returns'] = df[config['data']['target_column']].pct_change()
plt.figure(figsize=(14, 4))
sns.histplot(x=df['Returns'].dropna(), bins=100, kde=True, color='purple')
plt.title("Distribution of Daily Returns (Checking for Fat Tails)")
plt.show()

print(f"Dataset Statistics:\n{df[config['data']['target_column']].describe()}")

Data Preprocessing

In [ ]:
# Initialize Processor
dp = DataProcessor(window=config['data']['window_size'])

# Scale and Sequence
scaled_data = dp.load_and_scale(f"../{config['data']['file_path']}")
X, y = dp.create_sequences(scaled_data)

# Train/Calibration/Test Split
(X_train, y_train), (X_cal, y_cal), (X_test, y_test) = dp.split_data(
    X, y, 
    train_p=config['data']['train_split'], 
    cal_p=config['data']['cal_split']
)

print(f"Data Shapes:\nTrain: {X_train.shape} | Cal: {X_cal.shape} | Test: {X_test.shape}")

Model Training (Ensemble)

In [ ]:
print("--- Training LSTM Ensemble ---")
lstm_ens = EnsembleWrapper('LSTM', B=config['model']['ensemble_size'])
lstm_ens.fit(X_train, y_train, epochs=config['model']['epochs'])

print("--- Training GRU Ensemble ---")
gru_ens = EnsembleWrapper('GRU', B=config['model']['ensemble_size'])
gru_ens.fit(X_train, y_train, epochs=config['model']['epochs'])

Conformal Inference Logic

In [ ]:
# 1. Get Calibration Residuals (Crucial for Conformal Inference)
lstm_cal_pred = lstm_ens.predict(X_cal)
gru_cal_pred = gru_ens.predict(X_cal)

lstm_res = get_residuals(y_cal, lstm_cal_pred)
gru_res = get_residuals(y_cal, gru_cal_pred)

# 2. Generate Test Predictions
lstm_test_pred = lstm_ens.predict(X_test)
gru_test_pred = gru_ens.predict(X_test)

# 3. Apply EnbPI (Fixed Quantile)
l_enbpi, u_enbpi = apply_enbpi(
    lstm_test_pred,
    lstm_res,
    alpha=config['conformal']['alpha'],
)
g_enbpi, gu_enbpi = apply_enbpi(
    gru_test_pred,
    gru_res,
    alpha=config['conformal']['alpha'],
)

# 4. Apply AgACI (Rolling Window Adaptation)
l_agaci, u_agaci = rolling_agaci(
    lstm_test_pred,
    y_test,
    lstm_res,
    alpha=config['conformal']['alpha'],
    window=config['conformal']['agaci_window'],
)
g_agaci, gu_agaci = rolling_agaci(
    gru_test_pred,
    y_test,
    gru_res,
    alpha=config['conformal']['alpha'],
    window=config['conformal']['agaci_window'],
)

Final Visualization and Results

In [ ]:
# Plot 1: Forecast Comparison
plot_forecast_vs_actual(y_test, lstm_test_pred, gru_test_pred)

# Plot 2: Uncertainty Comparison
plot_prediction_intervals(y_test, lstm_test_pred, l_enbpi, u_enbpi, l_agaci, u_agaci, model_name="LSTM")

# Plot 3: Metrics Evaluation
enbpi_cov, enbpi_width = calculate_metrics(y_test, l_enbpi, u_enbpi)
agaci_cov, agaci_width = calculate_metrics(y_test, l_agaci, u_agaci)

# Summary Table
results_df = pd.DataFrame({
    "Method": ["EnbPI", "AgACI"],
    "Coverage (%)": [enbpi_cov * 100, agaci_cov * 100],
    "Avg Width": [enbpi_width, agaci_width]
})

print("\nFinal Conformal Inference Performance:")
print(results_df.to_string(index=False))

# Optional: Save results for later
# results_df.to_csv('../results/final_metrics.csv')